In [4]:
import os
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, AdamW, get_scheduler
import evaluate

# 1. SETUP & HYPERPARAMETERS
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "t5-base"
MAX_LEN = 512
SUMMARY_LEN = 64
BATCH_SIZE = 4
EPOCHS = 3 # Start with 3 to ensure stability
LEARNING_RATE = 5e-5

# 2. LOAD DATASET
print("Loading BBC News Dataset...")
dataset = load_dataset('gopalkalpande/bbc-news-summary', split='train')
full_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_data = full_dataset['train']
test_data = full_dataset['test']

# 3. INITIALIZE TOKENIZER & MODEL
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)

# 4. DATA PROCESSING FUNCTION
def collate_fn(batch):
    """Custom collate to handle padding manually."""
    inputs = ["summarize: " + item["Articles"] for item in batch]
    targets = [item["Summaries"] for item in batch]

    model_inputs = tokenizer(inputs, max_length=MAX_LEN, padding=True, truncation=True, return_tensors="pt")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=SUMMARY_LEN, padding=True, truncation=True, return_tensors="pt")

    # Replace padding token id (0) with -100 so it's ignored by the loss function
    labels_with_ignore = labels["input_ids"].masked_fill(labels["input_ids"] == tokenizer.pad_token_id, -100)

    return {
        "input_ids": model_inputs["input_ids"].to(DEVICE),
        "attention_mask": model_inputs["attention_mask"].to(DEVICE),
        "labels": labels_with_ignore.to(DEVICE)
    }

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, collate_fn=collate_fn)

# 5. OPTIMIZER & SCHEDULER
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
num_training_steps = EPOCHS * len(train_loader)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# 6. MANUAL TRAINING LOOP
print(f"Starting Training on {DEVICE}...")
model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        optimizer.zero_grad()

        # Forward Pass
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )

        loss = outputs.loss
        loss.backward()

        # Gradient Clipping (Prevents exploding gradients in Transformers)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}]")
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

# 7. EVALUATION (ROUGE)
print("\nRunning Evaluation...")
model.eval()
rouge = evaluate.load("rouge")

with torch.no_grad():
    for batch in tqdm(test_loader):
        generated_ids = model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=SUMMARY_LEN,
            num_beams=4
        )

        preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        # Clean labels back from -100 to 0 for decoding
        labels = batch["labels"].cpu().numpy()
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        refs = tokenizer.batch_decode(labels, skip_special_tokens=True)

        rouge.add_batch(predictions=preds, references=refs)

final_score = rouge.compute()
print("\nFinal ROUGE Metrics:", final_score)

# 8. SAVE
model.save_pretrained("./mfc3_final_model")
tokenizer.save_pretrained("./mfc3_final_model")
print("Model saved to ./mfc3_final_model")

Loading BBC News Dataset...


Generating train split: 0 examples [00:00, ? examples/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Epoch 2 Average Loss: 0.4625


  0%|          | 0/501 [00:00<?, ?it/s]

Epoch 3 Average Loss: 0.4183

Running Evaluation...


  0%|          | 0/56 [00:00<?, ?it/s]


Final ROUGE Metrics: {'rouge1': np.float64(0.5487119701687928), 'rouge2': np.float64(0.4305658697290544), 'rougeL': np.float64(0.4510252793855477), 'rougeLsum': np.float64(0.44923565163581286)}
Model saved to ./mfc3_final_model


In [5]:
!pip install -q gradio

import gradio as gr
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

# 1. Load the model you just saved
model_path = "./mfc3_final_model"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = T5ForConditionalGeneration.from_pretrained(model_path).to(device)
tokenizer = T5Tokenizer.from_pretrained(model_path)

# 2. Define the Inference Function
def summarize_news(text):
    if not text.strip():
        return "Please enter some text to summarize."

    # Preprocess (Add the T5 prefix)
    input_text = "summarize: " + text
    inputs = tokenizer.encode(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    # Generate using Beam Search (Matches your Stage 7 logic)
    summary_ids = model.generate(
        inputs,
        max_length=150,
        num_beams=5,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

    output = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return output

# 3. Create the Gradio Web Interface (Your "HTML Page")
interface = gr.Interface(
    fn=summarize_news,
    inputs=gr.Textbox(lines=10, placeholder="Paste BBC News Article here...", label="Input Article"),
    outputs=gr.Textbox(label="AI Generated Summary"),
    title="MFC3: Computationally Efficient Summarizer",
    description="Group D1: Text Summarization using Fine-Tuned T5-Base. Powered by Amrita Vishwa Vidyapeetham.",
    theme="soft"
)

# 4. Launch with a public link
interface.launch(share=True)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://96846743a8b8f84017.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
